# Session 20 — MLOps for Medical Image Analysis using Deep Learning

**Goal:** the deep-learning-on-images counterpart to Session 15's tabular healthcare
pipeline — train a small CNN on medical-style images, track it with MLflow, and
serve it, using a fully local, fully runnable dataset (scikit-learn's `digits`,
standing in for e.g. X-ray patches — see the note below).

## A note on the dataset

Real medical imaging (chest X-rays, histopathology slides, MRI scans) needs
specialized public datasets (NIH ChestX-ray14, CheXpert) that aren't available in
this sandbox and usually require a data-use agreement. This notebook uses
scikit-learn's `digits` dataset (8x8 grayscale images) purely as a structurally
similar stand-in — a small image, one label per image — so the full CNN-training and
MLOps-tracking pipeline can run end to end locally. Swap in a real medical dataset by
changing only Step 1's data loading.

## Prerequisites

```bash
pip install torch mlflow
```
Runs entirely locally on CPU (the model and dataset are intentionally tiny).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import mlflow
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
mlflow.set_tracking_uri("mlruns")
mlflow.set_experiment("session20-medical-imaging")

## Step 1 — Load and prepare "medical-style" images

Reshaped to `(N, 1, 8, 8)` — one grayscale channel, matching how a real X-ray patch
tensor would be shaped for a CNN (`(N, channels, height, width)`).

In [ ]:
digits = load_digits()
X = digits.images.astype(np.float32) / 16.0  # normalize to [0, 1]
X = X.reshape(-1, 1, 8, 8)
y = (digits.target >= 5).astype(np.int64)  # binarize: stand-in for "abnormal vs normal" finding

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
test_ds = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64)

print(f"train: {len(train_ds)}, test: {len(test_ds)}, positive rate: {y.mean():.2%}")

## Step 2 — A small CNN

Two convolutional blocks followed by a classifier head — the same architectural
pattern (conv -> pool -> conv -> pool -> flatten -> linear) used by real medical
imaging CNNs, just far smaller since our images are 8x8 instead of 224x224.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(32 * 2 * 2, 64)
        self.fc2 = nn.Linear(64, 2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))   # 8x8 -> 4x4
        x = self.pool(self.relu(self.conv2(x)))   # 4x4 -> 2x2
        x = x.flatten(1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

model = SmallCNN()
n_params = sum(p.numel() for p in model.parameters())
print(f"model has {n_params:,} parameters")

## Step 3 — Train, tracking every epoch's metrics with MLflow

Same tracking discipline as every other session: log hyperparameters up front, log a
metric per epoch (not just the final number) so training curves are inspectable
later, log the final model as an artifact.

In [ ]:
LR = 0.001
EPOCHS = 15

optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

with mlflow.start_run(run_name="medical_cnn") as run:
    mlflow.log_param("lr", LR)
    mlflow.log_param("epochs", EPOCHS)
    mlflow.log_param("architecture", "SmallCNN (2 conv blocks)")

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        mlflow.log_metric("train_loss", avg_loss, step=epoch)

        if epoch % 5 == 0 or epoch == EPOCHS - 1:
            print(f"epoch {epoch+1}/{EPOCHS}  loss={avg_loss:.4f}")

    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
            all_preds.extend(preds.tolist())
            all_labels.extend(yb.tolist())

    test_acc = correct / total
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.pytorch.log_model(model, artifact_path="model")

    cnn_run_id = run.info.run_id
    print(f"\ntest accuracy: {test_acc:.4f}")

## Step 4 — Per-class performance: don't trust one aggregate number

A false negative (missing an actual abnormal finding) is usually far costlier than a
false positive in a medical context — always check the confusion matrix, not just
overall accuracy.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(all_labels, all_preds)
print("Confusion matrix (rows=actual, cols=predicted):")
print(cm)
print()
print(classification_report(all_labels, all_preds, target_names=["normal", "abnormal"]))

## Step 5 — Serve the model

Same FastAPI pattern as every other serving session in this course — the interface
doesn't change just because the model underneath is a CNN instead of a
RandomForest.

In [ ]:
import mlflow.pytorch
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import List

loaded_cnn = mlflow.pytorch.load_model(f"runs:/{cnn_run_id}/model")
loaded_cnn.eval()

app = FastAPI(title="Medical Image Screening API")

class ImageRequest(BaseModel):
    pixels: List[List[float]]  # 8x8 grid, values in [0, 1]

@app.post("/v1/screen-image")
def screen_image(request: ImageRequest):
    x = torch.tensor(request.pixels, dtype=torch.float32).reshape(1, 1, 8, 8)
    with torch.no_grad():
        logits = loaded_cnn(x)
        proba = torch.softmax(logits, dim=1)[0, 1].item()
    return {"abnormal_probability": round(proba, 4), "flagged_for_review": proba > 0.5}

client = TestClient(app)
sample_pixels = X_test[0, 0].tolist()
response = client.post("/v1/screen-image", json={"pixels": sample_pixels})
print(response.status_code, response.json())

## What to try next

* Swap in a real dataset (e.g. NIH ChestX-ray14 after signing its data-use
  agreement) and a deeper pretrained backbone (`torchvision.models.resnet18` with
  transfer learning) — the training loop, MLflow tracking, and serving code above
  need essentially no changes.
* Add Grad-CAM visualization so a flagged image comes with a heatmap showing *which*
  region drove the "abnormal" prediction — critical for clinician trust, complementing
  the tabular SHAP explanations in Session 22.
* Add the Deepchecks image-domain checks (label distribution, near-duplicate
  detection) analogous to Session 11's tabular checks, since medical imaging datasets
  are especially prone to patient-level data leakage between train/test splits.